# Idea
{high level idea here}

# Experiment
1. Try with a few samples
2. Evaluate dev score

In [8]:
# Helper functions
import base64
import os
import random
from typing import List
from PIL import Image
from openai import OpenAI


VLM_MODEL_ID = "openrouter/openai/gpt-5.1-chat"
LLM_MODEL_ID = "openrouter/openai/gpt-5.1-chat"
openai_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

'''
Sample n flowchart images
Inputs:
    - image folder path
    - n
    - boolean indicating whether to display the sampled images
    - random seed
Outputs:
    - list of PIL Image objects
'''
def sample_n_images(image_folder: str, n: int, display_image: bool = False, seed: int | None = None) -> List[Image.Image]:
    if seed is not None:
        random.seed(seed)

    # Supported image extensions
    image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".webp"}

    # Collect image paths
    image_paths = [
        os.path.join(image_folder, f)
        for f in os.listdir(image_folder)
        if os.path.splitext(f.lower())[1] in image_exts
    ]

    if len(image_paths) == 0:
        raise ValueError(f"No image files found in directory: {image_folder}")

    if n > len(image_paths):
        raise ValueError(f"Requested n={n} images, but only {len(image_paths)} available")

    # Sample image paths
    sampled_paths = random.sample(image_paths, n)

    # Load images
    images = [Image.open(p).convert("RGB") for p in sampled_paths]

    # Optionally display
    if display_image:
        for img, path in zip(images, sampled_paths):
            print(os.path.basename(path))
            img.show()

    return images


'''
Query the VLM with image and a text prompt.
Inputs:
    - image file path
    - text prompt
Outputs:
    - response by the VLM
'''
def query_VLM(image_path, prompt, max_new_tokens=4096, temperature=0.0):
    # encode as base64 binary
    with open(image_path, "rb") as image_file:
        image =  base64.b64encode(image_file.read()).decode("utf-8")

    messages = [
                {"role": "system", "content": "You are a helpful assistant."},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{image}",
                                "detail": "high",
                            },
                        },
                    ],
                },
            ]

    completion = client.chat.completions.create(
        model=VLM_MODEL_ID,
        max_tokens=max_new_tokens,
        temperature=temperature,
        messages=messages,
    )
    return completion

'''
Query the LLM with a text prompt.
Inputs:
    - text prompt
Outputs:
    - response by the LLM
'''
def query_LLM(prompt, max_new_tokens=4096, temperature=0.0):
    messages = [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt},
            ]

    completion = client.chat.completions.create(
        model=LLM_MODEL_ID,
        max_tokens=max_new_tokens,
        temperature=temperature,
        messages=messages,
    )
    return completion

# Dev score